## Load Data

In [26]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel("Football_Dataset.xlsx")

df.head()

,PlayerID,PlayerName,Team,Position,MatchDate,Opponent,MinutesPlayed,Goals,Assists,Shots,...,KeyPasses,Dribbles,PassAccuracy,Tackles,Interceptions,Clearances,FoulsCommitted,YellowCards,RedCards,Rating
0,4591,Player_38,Brighton,Forward,2021-08-18,Aston Villa,80,3,1,6,...,3,2,68,1,3,1,2,0,1,6.1
1,3091,Player_10,Aston Villa,Defender,2022-01-12,Chelsea,86,3,1,4,...,4,4,87,0,2,4,1,1,1,8.2
2,9023,Player_27,Aston Villa,Defender,2021-10-13,Manchester City,69,2,0,4,...,4,1,73,5,3,7,3,0,1,6.2
3,9899,Player_86,Newcastle,Forward,2022-04-24,Crystal Palace,69,3,1,7,...,5,4,80,2,5,4,3,2,1,7.6
4,5506,Player_113,Leeds United,Midfielder,2022-01-18,Brighton,80,2,2,2,...,1,4,91,5,1,6,2,1,0,7.4


## Data Cleaning and Preprocessing

In [27]:
# Check data types and missing values
df.info()

# Convert MatchDate to datetime objects
df["MatchDate"] = pd.to_datetime(df["MatchDate"])

# Remove rows with missing values and eliminate duplicate entries
df = df.dropna().drop_duplicates()

# Summary of null values after cleaning
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PlayerID        5000 non-null   int64  
 1   PlayerName      5000 non-null   object 
 2   Team            5000 non-null   object 
 3   Position        5000 non-null   object 
 4   MatchDate       5000 non-null   object 
 5   Opponent        5000 non-null   object 
 6   MinutesPlayed   5000 non-null   int64  
 7   Goals           5000 non-null   int64  
 8   Assists         5000 non-null   int64  
 9   Shots           5000 non-null   int64  
 10  xG              5000 non-null   float64
 11  xA              5000 non-null   float64
 12  KeyPasses       5000 non-null   int64  
 13  Dribbles        5000 non-null   int64  
 14  PassAccuracy    5000 non-null   int64  
 15  Tackles         5000 non-null   int64  
 16  Interceptions   5000 non-null   int64  
 17  Clearances      5000 non-null   i

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   PlayerID        5000 non-null   int64         
 1   PlayerName      5000 non-null   object        
 2   Team            5000 non-null   object        
 3   Position        5000 non-null   object        
 4   MatchDate       5000 non-null   datetime64[ns]
 5   Opponent        5000 non-null   object        
 6   MinutesPlayed   5000 non-null   int64         
 7   Goals           5000 non-null   int64         
 8   Assists         5000 non-null   int64         
 9   Shots           5000 non-null   int64         
 10  xG              5000 non-null   float64       
 11  xA              5000 non-null   float64       
 12  KeyPasses       5000 non-null   int64         
 13  Dribbles        5000 non-null   int64         
 14  PassAccuracy    5000 non-null   int64         
 15  Tack

## Main Data Aggregation For Player

In [3]:
# Aggregate raw metrics for all sections
player_stats = df.groupby("PlayerName").agg({
    "Goals": "sum", "Assists": "sum", "Shots": "sum", "xG": "sum", "xA": "sum",
    "KeyPasses": "sum", "Dribbles": "sum", "PassAccuracy": "mean", "Tackles": "sum",
    "Interceptions": "sum", "Clearances": "sum", "FoulsCommitted": "sum",
    "YellowCards": "sum", "RedCards": "sum", "Rating": "mean",
    "MinutesPlayed": "sum", "MatchDate": "nunique", "Team": "first"
}).reset_index()

player_stats.rename(columns={"MatchDate": "Matches", "Rating": "Avg_Rating"}, inplace=True)

# Global Team Metrics
team_metrics = df.groupby("Team").agg({"Goals": "sum", "KeyPasses": "sum", "MinutesPlayed": "sum"}).reset_index()
team_metrics.columns = ["Team", "Team_Total_Goals", "Team_Total_KP", "Team_Total_Minutes"]
player_stats = player_stats.merge(team_metrics, on="Team", how="left")

# Helper Counts (Matches with actions)
matches_with_goals = df[df["Goals"] > 0].groupby("PlayerName")["MatchDate"].nunique().reset_index()
matches_with_goals.columns = ["PlayerName", "Matches_With_Goals"]

gc_matches = df[(df["Goals"] > 0) | (df["Assists"] > 0)].groupby("PlayerName")["MatchDate"].nunique().reset_index()
gc_matches.columns = ["PlayerName", "GC_Matches"]

creative_matches = df[df["KeyPasses"] > 0].groupby("PlayerName")["MatchDate"].nunique().reset_index()
creative_matches.columns = ["PlayerName", "Creative_Matches"]

# Variance & Standard Deviations
stds = df.groupby("PlayerName").agg({"Goals": "var", "PassAccuracy": "std", "Tackles": "std", "MinutesPlayed": "std"}).reset_index()
stds.columns = ["PlayerName", "Goals_Variance", "Pass_Std", "Tackle_Std", "Minutes_Std"]

# Merge helpers
player_stats = player_stats.merge(matches_with_goals, on="PlayerName", how="left")
player_stats = player_stats.merge(gc_matches, on="PlayerName", how="left")
player_stats = player_stats.merge(creative_matches, on="PlayerName", how="left")
player_stats = player_stats.merge(stds, on="PlayerName", how="left")
player_stats.fillna(0, inplace=True)

A. Goal Scoring KPIs

In [4]:
# Average goals per match and per 90 minutes played
player_stats["Goals_per_Match"] = player_stats["Goals"] / player_stats["Matches"]
player_stats["Goals_per_90"] = (player_stats["Goals"] / player_stats["MinutesPlayed"]) * 90

# Shot quality and finishing accuracy
player_stats["Shot_Conversion"] = player_stats["Goals"] / (player_stats["Shots"] + 0.1)
player_stats["xG_Efficiency"] = player_stats["Goals"] - player_stats["xG"]
player_stats["Finishing_Efficiency"] = player_stats["Goals"] / (player_stats["xG"] + 0.1)

# Team reliance and scoring stability
player_stats["Goal_Participation"] = player_stats["Goals"] / (player_stats["Team_Total_Goals"] + 0.1)
player_stats["Goal_Consistency"] = player_stats["Matches_With_Goals"] / player_stats["Matches"]
player_stats["Finishing_vs_Volume"] = player_stats["Goals_per_90"] / ((player_stats["Shots"] / player_stats["MinutesPlayed"]) * 90 + 0.1)

B. Goal Contribution KPIs

In [5]:
# Total direct involvements (Goals + Assists) and their rates
player_stats["Goal_Contribution"] = player_stats["Goals"] + player_stats["Assists"]
player_stats["GC_per_Match"] = player_stats["Goal_Contribution"] / player_stats["Matches"]
player_stats["GC_per_90"] = (player_stats["Goal_Contribution"] / player_stats["MinutesPlayed"]) * 90

# Efficiency of actual assists relative to expected assists (xA)
player_stats["Assist_Efficiency"] = player_stats["Assists"] / (player_stats["xA"] + 0.1)

# Direct contribution share of the team's total goals
player_stats["GC_Percentage"] = player_stats["Goal_Contribution"] / (player_stats["Team_Total_Goals"] + 0.1)

# Weighted score for overall attacking impact
player_stats["Creative_Impact_B"] = (player_stats["Goals"] * 0.6) + (player_stats["Assists"] * 0.8) + (player_stats["xA"] * 0.4)

# Frequency of goal contribution across matches
player_stats["Contribution_Consistency"] = player_stats["GC_Matches"] / player_stats["Matches"]

C. Shooting Quality KPIs

In [6]:
# Quality of chances selected (Average xG per shot)
player_stats["xG_per_Shot"] = player_stats["xG"] / (player_stats["Shots"] + 0.1)

# Direct output per shot (Goals + Assists per shot taken)
player_stats["Shot_Productivity"] = (player_stats["Goals"] + player_stats["Assists"]) / (player_stats["Shots"] + 0.1)

# Composite score for finishing intelligence and shot value
player_stats["Finishing_Intelligence"] = (
    (player_stats["Goals"] / (player_stats["Shots"] + 0.1)) * 0.5 + 
    (player_stats["Goals"] / (player_stats["xG"] + 0.1)) * 0.3 + 
    (player_stats["xG_per_Shot"] * 0.2)
)

# Total weighted shooting impact
player_stats["Shooting_Impact"] = (
    (player_stats["Goals"] * 0.6) + 
    ((player_stats["Goals"] / (player_stats["Shots"] + 0.1)) * 0.2) + 
    (player_stats["Shot_Productivity"] * 0.2)
)

D. Creativity & Playmaking KPIs

In [7]:
# Key passes frequency and dribbling volume
player_stats["KeyPasses_per_90"] = (player_stats["KeyPasses"] / player_stats["MinutesPlayed"]) * 90
player_stats["Dribbles_per_Match"] = player_stats["Dribbles"] / player_stats["Matches"]
player_stats["Chance_Creation_Index"] = player_stats["KeyPasses"] + player_stats["Assists"]

# Ability to convert key passes into actual assists
player_stats["Creativity_Efficiency"] = player_stats["Assists"] / (player_stats["KeyPasses"] + 1)

# Player's share of team's total playmaking output
player_stats["Playmaking_Load"] = player_stats["KeyPasses"] / (player_stats["Team_Total_KP"] + 0.1)

# Consistency of creativity across all matches
player_stats["Creative_Consistency"] = player_stats["Creative_Matches"] / player_stats["Matches"]

# The impact of dribbling on creating clear chances
player_stats["Dribble_Impact"] = player_stats["Dribbles_per_Match"] * player_stats["Chance_Creation_Index"]

E. Passing Performance KPIs

In [8]:
# General accuracy and stability of passing across matches
player_stats["Avg_Pass_Accuracy"] = player_stats["PassAccuracy"]
player_stats["Passing_Consistency"] = player_stats["Avg_Pass_Accuracy"] / (player_stats["Pass_Std"] + 1)

# Weighted passing impact score
player_stats["Passing_Impact"] = (player_stats["Avg_Pass_Accuracy"] * 0.5) + (player_stats["Passing_Consistency"] * 0.5)

# Total possession value based on accuracy and volume
player_stats["Possession_Value"] = (player_stats["Avg_Pass_Accuracy"] * player_stats["MinutesPlayed"] / 100)

F. Defensive Performance KPIs

In [9]:
# Defensive actions volume per match and normalized per 90
player_stats["Tackles_per_Match"] = player_stats["Tackles"] / player_stats["Matches"]
player_stats["Defensive_Actions"] = player_stats["Tackles"] + player_stats["Interceptions"] + player_stats["Clearances"]
player_stats["Def_Actions_per_90"] = (player_stats["Defensive_Actions"] / player_stats["MinutesPlayed"]) * 90

# Overall defensive impact and activity intensity
player_stats["Defensive_Impact_F"] = (player_stats["Tackles"] * 0.4) + (player_stats["Interceptions"] * 0.3) + (player_stats["Clearances"] * 0.3)
player_stats["Defensive_Activity"] = player_stats["Def_Actions_per_90"] * (player_stats["Defensive_Actions"] / (player_stats["MinutesPlayed"] + 0.1))

G. Defensive Impact KPIs (Advanced)

In [10]:
# Measures the ability to win the ball without committing fouls
player_stats["Clean_Defense_Index"] = (player_stats["Tackles"] + player_stats["Interceptions"]) / (player_stats["FoulsCommitted"] + 1)

# Ratio of fouls to defensive interventions
player_stats["Aggression_Control_G"] = player_stats["FoulsCommitted"] / (player_stats["Tackles"] + player_stats["Interceptions"] + 1)

# General defensive discipline score
player_stats["Def_Discipline"] = 1 / (player_stats["FoulsCommitted"] + 1)

H. Discipline KPIs

In [11]:
# Card frequency and total discipline burden (Weighting cards)
player_stats["Card_Rate"] = (player_stats["YellowCards"] + player_stats["RedCards"]) / player_stats["Matches"]
player_stats["Discipline_Score"] = player_stats["FoulsCommitted"] + player_stats["YellowCards"] + (player_stats["RedCards"] * 2)

# Fouls committed for every card received
player_stats["Fouls_to_Cards"] = player_stats["FoulsCommitted"] / (player_stats["YellowCards"] + player_stats["RedCards"] + 1)

# Overall clean play indicator
player_stats["Clean_Play"] = 1 / (player_stats["Discipline_Score"] + 1)

I. Playing Time KPIs

In [12]:
# Share of team minutes played and match involvement
player_stats["Usage_Rate"] = player_stats["MinutesPlayed"] / (player_stats["Team_Total_Minutes"] + 0.1)
player_stats["Match_Involvement"] = player_stats["Matches"] / (player_stats["Matches"].max() + 0.1)

# Stability of playing time (Standard deviation check)
player_stats["Playing_Consistency"] = player_stats["MinutesPlayed"] / (player_stats["Minutes_Std"] + 1)

# General availability based on match appearance and minutes
player_stats["Availability_Score"] = player_stats["Matches"] * (player_stats["MinutesPlayed"] / (player_stats["Matches"] + 0.1))

J. Overall Performance KPIs

In [13]:
# Combined indexes for impact and overall contribution
player_stats["Performance_Index"] = player_stats["Avg_Rating"] + player_stats["Goals"] + player_stats["Assists"]
player_stats["Overall_Contribution"] = player_stats["Goals"] + player_stats["Assists"] + player_stats["KeyPasses"] + player_stats["Tackles"] + player_stats["Interceptions"]

# Weighted score for identifying the 'Star' impact
player_stats["Star_Index"] = player_stats["Avg_Rating"] * (player_stats["Goal_Contribution"] + 1)

# Final balanced score covering both offense and defense
player_stats["Balanced_Performance"] = ( (player_stats["Goal_Contribution"] * 0.5) + (player_stats["Defensive_Impact_F"] * 0.5) )

## Main Data Aggregation For Team

In [14]:
## Load Data & Preprocessing for Teams
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel("Football_Dataset.xlsx")

# Data Cleaning
df["MatchDate"] = pd.to_datetime(df["MatchDate"])
df = df.dropna().drop_duplicates()

# Create a copy for team aggregation
team_df = df.copy()

In [15]:
## Main Data Aggregation for Teams
team_stats = team_df.groupby("Team").agg({
    "Goals": "sum",
    "Assists": "sum",
    "Shots": "sum",
    "KeyPasses": "sum",
    "Dribbles": "sum",
    "PassAccuracy": "mean",
    "Tackles": "sum",
    "Interceptions": "sum",
    "Clearances": "sum",
    "FoulsCommitted": "sum",
    "YellowCards": "sum",
    "RedCards": "sum",
    "Rating": "mean",
    "MinutesPlayed": "sum",
    "MatchDate": "nunique",
    "PlayerName": "nunique"  
}).reset_index()

# Rename columns for clarity
team_stats.rename(columns={
    "MatchDate": "Matches_Played",
    "PlayerName": "Unique_Players",
    "Rating": "Avg_Team_Rating",
    "PassAccuracy": "Avg_Pass_Accuracy"
}, inplace=True)

# Calculate Total Defensive Actions
team_stats["Total_Defensive_Actions"] = team_stats["Tackles"] + team_stats["Interceptions"] + team_stats["Clearances"]

# Calculate Goal Contribution (Goals + Assists)
team_stats["Team_Goal_Contribution"] = team_stats["Goals"] + team_stats["Assists"]

print("Team Data Aggregated Successfully!")

Team Data Aggregated Successfully!


A. Team Attack KPIs

In [16]:
## A. Team Attack KPIs

# Average Goals per Player
team_stats["Avg_Goals_per_Player"] = team_stats["Goals"] / team_stats["Unique_Players"]

# Total Team Goals (already exists)
# Total Team Assists (already exists)
# Team Goal Contribution (already exists)

# Team Shot Volume (already exists as "Shots")

# Team Shot Efficiency (Goals per Shot)
team_stats["Team_Shot_Efficiency"] = team_stats["Goals"] / (team_stats["Shots"] + 0.1)

# Team Goals per Match
team_stats["Team_Goals_per_Match"] = team_stats["Goals"] / team_stats["Matches_Played"]

# Team Assists per Match
team_stats["Team_Assists_per_Match"] = team_stats["Assists"] / team_stats["Matches_Played"]

print("Team Attack KPIs Added!")

Team Attack KPIs Added!


B. Team Creativity KPIs

In [17]:
## B. Team Creativity KPIs

# Total Key Passes by Team (already exists)
# Total Dribbles by Team (already exists)

# Average Key Passes per Player
team_stats["Avg_KeyPasses_per_Player"] = team_stats["KeyPasses"] / team_stats["Unique_Players"]

# Average Dribbles per Player
team_stats["Avg_Dribbles_per_Player"] = team_stats["Dribbles"] / team_stats["Unique_Players"]

# Key Passes per Match
team_stats["KeyPasses_per_Match"] = team_stats["KeyPasses"] / team_stats["Matches_Played"]

# Dribbles per Match
team_stats["Dribbles_per_Match"] = team_stats["Dribbles"] / team_stats["Matches_Played"]

# Creativity Index (Combining Key Passes and Dribbles)
team_stats["Creativity_Index"] = (team_stats["KeyPasses"] * 0.6) + (team_stats["Dribbles"] * 0.4)

print("Team Creativity KPIs Added!")

Team Creativity KPIs Added!


C. Team Passing KPIs

In [18]:
## C. Team Passing KPIs

# Average Team Pass Accuracy (already exists as "Avg_Pass_Accuracy")

# Passing Consistency Index (based on variation - we need std per team)
# We need to calculate std of PassAccuracy per team from original data
pass_std = team_df.groupby("Team")["PassAccuracy"].std().reset_index()
pass_std.columns = ["Team", "Pass_Accuracy_Std"]

# Merge with team_stats
team_stats = team_stats.merge(pass_std, on="Team", how="left")
team_stats["Pass_Accuracy_Std"].fillna(0, inplace=True)

# Passing Consistency Index (Higher is better - low std means consistent)
team_stats["Passing_Consistency_Index"] = team_stats["Avg_Pass_Accuracy"] / (team_stats["Pass_Accuracy_Std"] + 1)

# Passing Volume per Match
team_stats["Passing_Volume"] = (team_stats["Avg_Pass_Accuracy"] * team_stats["Matches_Played"]) / 100

print("Team Passing KPIs Added!")

Team Passing KPIs Added!


C:\Users\AA\AppData\Local\Temp\ipykernel_12784\815215631.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  team_stats["Pass_Accuracy_Std"].fillna(0, inplace=True)


D. Team Defense KPIs

In [ ]:
## D. Team Defense KPIs

# Total Tackles by Team (already exists)
# Total Interceptions by Team (already exists)
# Total Clearances by Team (already exists)
# Total Defensive Actions (already calculated)

# Defensive Actions per Match
team_stats["Def_Actions_per_Match"] = team_stats["Total_Defensive_Actions"] / team_stats["Matches_Played"]

# Defensive Actions per Player (Average)
team_stats["Def_Actions_per_Player"] = team_stats["Total_Defensive_Actions"] / team_stats["Unique_Players"]

# Weighted Defensive Impact
team_stats["Defensive_Impact"] = (
    (team_stats["Tackles"] * 0.4) + 
    (team_stats["Interceptions"] * 0.35) + 
    (team_stats["Clearances"] * 0.25)
)

# Tackles per Match
team_stats["Tackles_per_Match"] = team_stats["Tackles"] / team_stats["Matches_Played"]

# Interceptions per Match
team_stats["Interceptions_per_Match"] = team_stats["Interceptions"] / team_stats["Matches_Played"]

# Clearances per Match
team_stats["Clearances_per_Match"] = team_stats["Clearances"] / team_stats["Matches_Played"]

print("Team Defense KPIs Added!")

Team Defense KPIs Added!


E. Team Discipline KPIs

In [ ]:
## E. Team Discipline KPIs

# Total Fouls by Team (already exists)
# Total Yellow Cards (already exists)
# Total Red Cards (already exists)

# Team Card Rate (Cards per Match)
team_stats["Team_Card_Rate"] = (team_stats["YellowCards"] + team_stats["RedCards"]) / team_stats["Matches_Played"]

# Fouls per Match
team_stats["Fouls_per_Match"] = team_stats["FoulsCommitted"] / team_stats["Matches_Played"]

# Discipline Score (Weighted)
team_stats["Team_Discipline_Score"] = team_stats["FoulsCommitted"] + (team_stats["YellowCards"] * 2) + (team_stats["RedCards"] * 3)

# Clean Play Index (Higher is better)
team_stats["Clean_Play_Index"] = 1 / (team_stats["Team_Discipline_Score"] + 1)

# Yellow Cards per Match
team_stats["Yellow_per_Match"] = team_stats["YellowCards"] / team_stats["Matches_Played"]

# Red Cards per Match
team_stats["Red_per_Match"] = team_stats["RedCards"] / team_stats["Matches_Played"]

print("Team Discipline KPIs Added!")

Team Discipline KPIs Added!


F. Team Overall Performance KPIs

In [ ]:
# Total Team Impact Score (Weighted combination)
team_stats["Team_Impact_Score"] = (
    (team_stats["Goals"] * 0.5) +
    (team_stats["Assists"] * 0.3) +
    (team_stats["KeyPasses"] * 0.15) +
    (team_stats["Total_Defensive_Actions"] * 0.05)
)

# Team Contribution Index = Sum(Goals + Assists + KeyPasses)
team_stats["Team_Contribution_Index"] = team_stats["Goals"] + team_stats["Assists"] + team_stats["KeyPasses"]

# Performance per Match
team_stats["Team_Performance_per_Match"] = team_stats["Team_Contribution_Index"] / team_stats["Matches_Played"]

# Star Team Index (Rating * Contribution)
team_stats["Star_Team_Index"] = team_stats["Avg_Team_Rating"] * (team_stats["Team_Goal_Contribution"] + 1)

# Balanced Team Score (Attack + Defense)
# Normalize each component first (Min-Max or simple z-score approach - using simple normalization here)
team_stats["Balanced_Team_Score"] = (
    (team_stats["Team_Goal_Contribution"] / (team_stats["Team_Goal_Contribution"].max() + 0.1)) * 0.5 +
    (team_stats["Total_Defensive_Actions"] / (team_stats["Total_Defensive_Actions"].max() + 0.1)) * 0.5
)

print("Team Overall Performance KPIs Added!")

Team Overall Performance KPIs Added!


## Export Final Table

In [31]:
# Clean Player Stats
player_stats.replace([np.inf, -np.inf], 0, inplace=True)
player_stats.fillna(0, inplace=True)

# Clean Team Stats
team_stats.replace([np.inf, -np.inf], 0, inplace=True)
team_stats.fillna(0, inplace=True)

# Create Excel writer object
with pd.ExcelWriter("Football_Final_Dataset.xlsx", engine='openpyxl') as writer:
    # Write Player Stats to Sheet 1
    player_stats.to_excel(writer, sheet_name="Players", index=False)
    
    # Write Team Stats to Sheet 2
    team_stats.to_excel(writer, sheet_name="Teams", index=False)

print("=" * 60)
print("✅ File Saved Successfully!")
print("=" * 60)
print(f"📊 Players Sheet: {len(player_stats)} rows, {len(player_stats.columns)} columns")
print(f"📊 Teams Sheet: {len(team_stats)} rows, {len(team_stats.columns)} columns")
print(f"📁 File name: Football_Final_Dataset.xlsx")
print("=" * 60)

print("\n📋 Players Preview (First 5 rows):")
print(player_stats[['PlayerName', 'Team', 'Goals', 'Assists', 'Avg_Rating']].head())

print("\n📋 Teams Preview (First 5 rows):")
print(team_stats[['Team', 'Goals', 'Assists', 'Matches_Played', 'Avg_Team_Rating']].head())

✅ File Saved Successfully!
📊 Players Sheet: 120 rows, 79 columns
📊 Teams Sheet: 20 rows, 48 columns
📁 File name: Football_Final_Dataset.xlsx

📋 Players Preview (First 5 rows):
   PlayerName          Team  Goals  Assists  Avg_Rating
0    Player_1       Chelsea     54       36    7.135484
1   Player_10   Aston Villa     62       26    7.351515
2  Player_100  Leeds United     60       41    7.319444
3  Player_101     Tottenham     54       29    7.337500
4  Player_102   Aston Villa     53       43    7.528571

📋 Teams Preview (First 5 rows):
          Team  Goals  Assists  Matches_Played  Avg_Team_Rating
0      Arsenal    344      231             167         7.389167
1  Aston Villa    428      286             172         7.285185
2    Brentford    353      236             156         7.270742
3     Brighton    377      239             174         7.418651
4      Burnley    361      234             163         7.167364


## Quick Summary of ALL KPIs

In [23]:
## Quick Summary of ALL KPIs (Players & Teams)

print("=" * 60)
print("PLAYER KPIs SUMMARY")
print("=" * 60)

# Player KPIs
player_kpi_columns = [col for col in player_stats.columns if col not in ["PlayerName", "Team", "Matches", "MinutesPlayed", "Team_Total_Goals", "Team_Total_KP", "Team_Total_Minutes"]]
print(f"Total Player KPIs: {len(player_kpi_columns)}")
print(f"First 5 Player KPIs: {player_kpi_columns[:5]}")

print("\n" + "=" * 60)
print("TEAM KPIs SUMMARY")
print("=" * 60)

# Team KPIs
team_kpi_columns = [col for col in team_stats.columns if col not in ["Team", "Matches_Played", "Unique_Players", "MinutesPlayed"]]
print(f"Total Team KPIs: {len(team_kpi_columns)}")
print(f"First 5 Team KPIs: {team_kpi_columns[:5]}")

print("\n" + "=" * 60)
print("SAMPLE DATA:")
print("-" * 40)
print("\nFirst 3 Players:")
print(player_stats[["PlayerName", "Team"] + player_kpi_columns[:5]].head(3))

print("\nFirst 3 Teams:")
print(team_stats[["Team"] + team_kpi_columns[:5]].head(3))

PLAYER KPIs SUMMARY
Total Player KPIs: 72
First 5 Player KPIs: ['Goals', 'Assists', 'Shots', 'xG', 'xA']

TEAM KPIs SUMMARY
Total Team KPIs: 44
First 5 Team KPIs: ['Goals', 'Assists', 'Shots', 'KeyPasses', 'Dribbles']

SAMPLE DATA:
----------------------------------------

First 3 Players:
   PlayerName          Team  Goals  Assists  Shots     xG     xA
0    Player_1       Chelsea     54       36    129  37.24  22.00
1   Player_10   Aston Villa     62       26    129  34.53  23.05
2  Player_100  Leeds United     60       41    119  47.88  28.65

First 3 Teams:
          Team  Goals  Assists  Shots  KeyPasses  Dribbles
0      Arsenal    344      231    828        571       637
1  Aston Villa    428      286    916        644       861
2    Brentford    353      236    803        619       742


## Push Data To SQL Server

In [24]:
## Complete Upload to SQL Server - Optimized Version
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import pyodbc

# ============================================
# 1. CONFIGURATION
# ============================================

server = 'DESKTOP-KB6OCRD\\SQLEXPRESS01'
database = 'Graduation_Project'

# ============================================
# 2. CONNECTION
# ============================================

print("🔌 Testing connection to SQL Server...")

try:
    # Test connection using pyodbc first
    conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes'
    test_conn = pyodbc.connect(conn_str)
    test_conn.close()
    print("✅ Connection successful!")
    
    # Create SQLAlchemy engine
    connection_string = f'mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
    engine = create_engine(connection_string)
    
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\n📌 Trying alternative server name...")
    
    try:
        # Try with localhost
        server = 'localhost\\SQLEXPRESS01'
        conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes'
        test_conn = pyodbc.connect(conn_str)
        test_conn.close()
        print("✅ Connection successful with localhost!")
        
        connection_string = f'mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
        engine = create_engine(connection_string)
        
    except Exception as e2:
        print(f"❌ All connection attempts failed: {e2}")
        print("\n📌 Please check:")
        print("1. SQL Server Express is running")
        print("2. Database 'Graduation_Project' exists")
        print("3. ODBC Driver 17 is installed")
        exit()

# ============================================
# 3. CLEAN DATA
# ============================================

print("\n🧹 Cleaning data...")

# Replace infinite values
player_stats.replace([np.inf, -np.inf], 0, inplace=True)
team_stats.replace([np.inf, -np.inf], 0, inplace=True)

# Fill NaN with 0
player_stats.fillna(0, inplace=True)
team_stats.fillna(0, inplace=True)

# Clean column names (remove special characters, limit length)
def clean_column_names(df):
    df.columns = [str(col).replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_').replace('+', 'plus').replace('-', '_').replace('%', 'pct').replace('&', 'and').replace('*', 'star') for col in df.columns]
    df.columns = [col[:128] for col in df.columns]  # SQL Server limit
    return df

player_stats = clean_column_names(player_stats)
team_stats = clean_column_names(team_stats)

# Convert float columns to proper types
for df in [player_stats, team_stats]:
    for col in df.select_dtypes(include=['float64', 'float32']).columns:
        df[col] = df[col].astype(float)

print(f"✅ Player stats: {len(player_stats)} rows, {len(player_stats.columns)} columns")
print(f"✅ Team stats: {len(team_stats)} rows, {len(team_stats.columns)} columns")

# ============================================
# 4. UPLOAD FUNCTION
# ============================================

def upload_table(df, table_name, engine):
    """Upload dataframe to SQL Server with fallback methods"""
    
    print(f"\n📤 Uploading {table_name}...")
    
    # Drop table if exists
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {table_name}"))
            conn.commit()
        print(f"🗑️ Old {table_name} table dropped")
    except:
        pass
    
    # Method 1: Try with small chunks (for tables with many columns)
    try:
        df.to_sql(
            table_name,
            engine,
            if_exists='replace',
            index=False,
            method='multi',
            chunksize=10  # Small chunks to avoid parameter limit
        )
        print(f"✅ {table_name} uploaded successfully (multi-insert method)!")
        return True
        
    except Exception as e:
        print(f"⚠️ Multi-insert failed: {str(e)[:100]}...")
        print("🔄 Trying single-insert method...")
        
        # Method 2: Single insert (slower but reliable)
        try:
            df.to_sql(
                table_name,
                engine,
                if_exists='replace',
                index=False,
                method=None
            )
            print(f"✅ {table_name} uploaded successfully (single-insert method)!")
            return True
            
        except Exception as e2:
            print(f"❌ All upload methods failed for {table_name}: {e2}")
            return False

# ============================================
# 5. UPLOAD DATA
# ============================================

print("\n" + "=" * 60)
print("📤 UPLOADING DATA TO SQL SERVER")
print("=" * 60)

# Upload Player Stats
success_player = upload_table(player_stats, 'PlayerStats', engine)

# Upload Team Stats
success_team = upload_table(team_stats, 'TeamStats', engine)

# ============================================
# 6. VERIFY UPLOAD
# ============================================

print("\n" + "=" * 60)
print("🔍 VERIFYING UPLOAD")
print("=" * 60)

try:
    with engine.connect() as conn:
        # List all tables
        tables = conn.execute(text("""
            SELECT TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE='BASE TABLE'
            ORDER BY TABLE_NAME
        """)).fetchall()
        
        table_list = [row[0] for row in tables]
        print(f"📋 Tables in database: {table_list}")
        
        # Verify PlayerStats
        if 'PlayerStats' in table_list:
            count = conn.execute(text("SELECT COUNT(*) FROM PlayerStats")).fetchone()[0]
            print(f"✅ PlayerStats: {count} rows")
            
            # Get column count
            cols = conn.execute(text("""
                SELECT COUNT(*) 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = 'PlayerStats'
            """)).fetchone()[0]
            print(f"✅ PlayerStats: {cols} columns")
        
        # Verify TeamStats
        if 'TeamStats' in table_list:
            count = conn.execute(text("SELECT COUNT(*) FROM TeamStats")).fetchone()[0]
            print(f"✅ TeamStats: {count} rows")
            
            cols = conn.execute(text("""
                SELECT COUNT(*) 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = 'TeamStats'
            """)).fetchone()[0]
            print(f"✅ TeamStats: {cols} columns")
    
    print("\n✅ Verification complete!")
    
except Exception as e:
    print(f"❌ Verification failed: {e}")

# ============================================
# 7. SAMPLE DATA PREVIEW
# ============================================

print("\n" + "=" * 60)
print("📋 SAMPLE DATA PREVIEW")
print("=" * 60)

try:
    # Preview PlayerStats
    if 'PlayerStats' in table_list:
        sample_player = pd.read_sql("SELECT TOP 3 PlayerName, Team, Goals, Assists, Matches FROM PlayerStats", engine)
        print("\n📊 PlayerStats (First 3 rows):")
        print(sample_player.to_string(index=False))
    
    # Preview TeamStats
    if 'TeamStats' in table_list:
        sample_team = pd.read_sql("SELECT TOP 3 Team, Goals, Assists, Matches_Played FROM TeamStats", engine)
        print("\n📊 TeamStats (First 3 rows):")
        print(sample_team.to_string(index=False))
        
except Exception as e:
    print(f"⚠️ Could not preview data: {e}")

# ============================================
# 8. CLOSE CONNECTION
# ============================================

engine.dispose()
print("\n" + "=" * 60)
print("✅ PROCESS COMPLETED SUCCESSFULLY!")
print("=" * 60)
print(f"📊 PlayerStats: {len(player_stats)} rows uploaded")
print(f"📊 TeamStats: {len(team_stats)} rows uploaded")

🔌 Testing connection to SQL Server...
✅ Connection successful!

🧹 Cleaning data...
✅ Player stats: 120 rows, 79 columns
✅ Team stats: 20 rows, 48 columns

📤 UPLOADING DATA TO SQL SERVER

📤 Uploading PlayerStats...


C:\Users\AA\AppData\Local\Temp\ipykernel_12784\1979297735.py:96: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


🗑️ Old PlayerStats table dropped
✅ PlayerStats uploaded successfully (multi-insert method)!

📤 Uploading TeamStats...
🗑️ Old TeamStats table dropped
✅ TeamStats uploaded successfully (multi-insert method)!

🔍 VERIFYING UPLOAD
📋 Tables in database: ['Football', 'Football_Full_Analysis', 'PlayerStats', 'TeamStats']
✅ PlayerStats: 120 rows
✅ PlayerStats: 79 columns
✅ TeamStats: 20 rows
✅ TeamStats: 48 columns

✅ Verification complete!

📋 SAMPLE DATA PREVIEW

📊 PlayerStats (First 3 rows):
PlayerName         Team  Goals  Assists  Matches
  Player_1      Chelsea     54       36       28
 Player_10  Aston Villa     62       26       33
Player_100 Leeds United     60       41       34

📊 TeamStats (First 3 rows):
       Team  Goals  Assists  Matches_Played
    Arsenal    344      231             167
Aston Villa    428      286             172
  Brentford    353      236             156

✅ PROCESS COMPLETED SUCCESSFULLY!
📊 PlayerStats: 120 rows uploaded
📊 TeamStats: 20 rows uploaded
